# Athletics Results Scraper
By Cesar Sierra. Learning project based on Alfredo Deza’s Coursera guided project.
Place the four course HTML files in `html/` before running all cells. The final section collects all results and verifies SQLite persistence.


# Persistence and Efficency
You now might be aware of some of the problems with parsing several pages and how that can quicly get out of hand. While parsing, you might chose to persist the collected data, so that it can be analyzed and cleaned later. 
Parsing, retrieving, saving, and cleaning data are all separate actions, and you shouldn't try to work with data while collecting. In this notebook you'll practice further parsing techniques along with persistency, both for JSON and CSV formats as well as using SQL and a database.

Start by loading one of the available HTML files into the `scrapy` library

**Note**: This notebook relies on several dependencies that must be pre-installed in your environment. If you haven't done so, please run the following command in your terminal or command prompt:
```bash
python3 -m venv venv
venv/bin/pip install -r requirements.txt
```

In [ ]:
import scrapy
import os

current_dir = os.path.abspath("")
file_path = os.path.join(
    current_dir,
    "html",
    "1992_World_Junior_Championships_in_Athletics_–_Men's_high_jump"
)

with open(file_path, encoding="utf-8") as html_file:
    html_data = html_file.read()

response = scrapy.http.TextResponse(
    url="https://example.org",
    body=html_data,
    encoding="utf-8"
)

print("HTML loaded successfully.")

In [ ]:
table = response.xpath("//table")[1].xpath("tbody")

for tr in table.xpath("tr"):
    medal = tr.xpath("td/b/text()").get()
    athlete = tr.xpath("td/a/text()").get()

    if medal and athlete:
        print(medal.strip(), athlete.strip())

In [ ]:
scraped_data = {}

for tr in table.xpath("tr"):
    medal = tr.xpath("td/b/text()").get()
    athlete = tr.xpath("td/a/text()").get()

    if medal and athlete:
        scraped_data[medal.strip()] = athlete.strip()

scraped_data

In [ ]:
import json

with open("1992_results.json", "w", encoding="utf-8") as json_file:
    json.dump(
        scraped_data,
        json_file,
        indent=4,
        ensure_ascii=False
    )

print("JSON saved.")

This interaction with `scrapy` in a Jupyter Notebook is useful because you don't need to run the special shell and you also don't need to run the whole spider. Once you learn what you need to do here, you can adapat the spider to persist data.
First, start by persisting data as JSON. To do this, you will need to keep the information in a Python data structure like a dictionary, and then load it as a JSON object, and finally, save it to a file.

In [ ]:
column_names = ["Medal", "Athlete"]
rows = []

for medal, athlete in scraped_data.items():
    rows.append([medal, athlete])

print(rows)

In [ ]:
import csv

with open(
    "1992_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(column_names)
    writer.writerows(rows)

print("CSV saved.")

In [ ]:
import sqlite3

connection = sqlite3.connect("1992_results.db")
cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS results (
        id INTEGER PRIMARY KEY,
        medal TEXT,
        athlete TEXT
    )
""")

connection.commit()
connection.close()

print("Database table ready.")

In [ ]:
connection = sqlite3.connect("1992_results.db")
cursor = connection.cursor()

cursor.execute("DELETE FROM results")

query = "INSERT INTO results (medal, athlete) VALUES (?, ?)"
cursor.executemany(query, rows)

connection.commit()
connection.close()

print(f"Saved {len(rows)} athletes.")

Finally, you can persist data to a database. Unlike the JSON and CSV approach, using a database is much more memory efficient. This is the principal reason why you want to use a database instead of a file on disk. Imagine capturing 10GB of data. This could potentially mean that you need 10GB of available memory to hold onto that data before saving it to disk.
By using a database, you can save the data as the data is gathered. 

For the next cells, use a SQLite database to persist the data. Create the file-based database and the table needed.

In [ ]:
connection = sqlite3.connect("1992_results.db")
cursor = connection.cursor()

cursor.execute("SELECT id, medal, athlete FROM results ORDER BY id")
saved_rows = cursor.fetchall()

for row in saved_rows:
    print(row)

connection.close()

In [ ]:
assert len(saved_rows) == len(rows), "The row counts do not match."

assert [list(row[1:]) for row in saved_rows] == rows, (
    "The saved data does not match the scraped data."
)

print(f"Verified: all {len(saved_rows)} records match.")

In [ ]:
print(os.listdir("html"))
print(response.xpath("//table")[1].get())

In [ ]:
# Show each table's first five rows
for index, html_table in enumerate(response.xpath("//table")):
    print(f"\nTABLE {index}")

    for tr in html_table.xpath(".//tr")[:5]:
        cells = tr.xpath("./th | ./td")

        values = [
            " ".join(cell.xpath(".//text()").getall()).strip()
            for cell in cells
        ]

        print(values)

In [ ]:
from pathlib import Path
import scrapy

all_results = []
file_counts = {}

def get_text(element):
    return " ".join(
        " ".join(element.xpath(".//text()").getall()).split()
    )

for file_path in sorted(Path("html").iterdir()):
    if not file_path.is_file():
        continue

    html_data = file_path.read_text(encoding="utf-8")

    page = scrapy.http.TextResponse(
        url=file_path.resolve().as_uri(),
        body=html_data,
        encoding="utf-8"
    )

    file_counts[file_path.name] = 0

    for table_index, html_table in enumerate(page.xpath("//table")):
        table_rows = html_table.xpath(".//tr")

        if not table_rows:
            continue

        headers = [
            get_text(cell).lower()
            for cell in table_rows[0].xpath("./th | ./td")
        ]

        required = ["rank", "name", "nationality", "result"]

        if not all(column in headers for column in required):
            continue

        for tr in table_rows[1:]:
            cells = tr.xpath("./th | ./td")

            if len(cells) < len(headers):
                continue

            values = [get_text(cell) for cell in cells]

            rank_cell = cells[headers.index("rank")]
            rank = values[headers.index("rank")]

            # Medal ranks may appear as images instead of text.
            if not rank:
                rank = rank_cell.xpath(".//img/@alt").get() or ""

            athlete = values[headers.index("name")]
            country = values[headers.index("nationality")]
            result = values[headers.index("result")]

            notes = (
                values[headers.index("notes")]
                if "notes" in headers
                else ""
            )

            if not athlete:
                continue

            all_results.append({
                "source_file": file_path.name,
                "table_index": table_index,
                "rank": rank,
                "athlete": athlete,
                "nationality": country,
                "result": result,
                "notes": notes
            })

            file_counts[file_path.name] += 1

for filename, count in file_counts.items():
    print(f"{filename}: {count} results")

print(f"\nTotal: {len(all_results)} results")

assert all_results, "No results found."
assert all(file_counts.values()), "A file had no matching results."

In [ ]:
import json
import csv

with open("all_results.json", "w", encoding="utf-8") as file:
    json.dump(all_results, file, indent=4, ensure_ascii=False)

column_names = [
    "source_file",
    "table_index",
    "rank",
    "athlete",
    "nationality",
    "result",
    "notes"
]

with open(
    "all_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:
    writer = csv.DictWriter(file, fieldnames=column_names)
    writer.writeheader()
    writer.writerows(all_results)

print("JSON and CSV saved.")

In [ ]:
import sqlite3

connection = sqlite3.connect("all_results.db")

try:
    with connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS results (
                id INTEGER PRIMARY KEY,
                source_file TEXT,
                table_index INTEGER,
                rank TEXT,
                athlete TEXT,
                nationality TEXT,
                result TEXT,
                notes TEXT
            )
        """)

        connection.execute("DELETE FROM results")

        connection.executemany("""
            INSERT INTO results (
                source_file, table_index, rank, athlete,
                nationality, result, notes
            )
            VALUES (
                :source_file, :table_index, :rank, :athlete,
                :nationality, :result, :notes
            )
        """, all_results)

finally:
    connection.close()

print(f"Saved {len(all_results)} results to SQLite.")

In [ ]:
connection = sqlite3.connect("all_results.db")
connection.row_factory = sqlite3.Row

try:
    saved_results = [
        dict(row)
        for row in connection.execute("""
            SELECT source_file, table_index, rank, athlete,
                   nationality, result, notes
            FROM results
            ORDER BY id
        """)
    ]
finally:
    connection.close()

assert saved_results == all_results, "Saved data does not match."

print(f"Verified: all {len(saved_results)} results match.")

for row in saved_results[:10]:
    print(row)

The data is now persisted in a file-based database that you can query. Verify that all works by creating a new connection and querying the database.

Update the _wikipedia_ project and spider to use some of these techniques to persist data. Next, try parsing all the files in the _html_ directory instead of just one and persist all results. Do you think you can parse other information as well? 

Try parsing the height and the results from all the other athletes, not just the top three places.